In [58]:
import pandas as pd
import numpy as np
import os
from scipy.signal import butter, filtfilt, convolve, welch

# --- CONFIGURACIÓN Y CONSTANTES ---
FS = 1259 # Frecuencia de muestreo (Hz)
LOW_CUT = 20.0
HIGH_CUT = 450.0
WINDOW_SIZE_MS = 100 
MVC_WINDOW_S = 0.5 # Ventana de tiempo para calcular el Max RMS del MVC (0.5 segundos)
ACTIVATION_THRESHOLD = 10 # Umbral de activación (10%)

TRIAL_FILES = ['trial_5.csv', 'trial_6.csv', 'trial_9.csv', 'trial_10.csv', 'trial_11.csv', 'trial_12.csv']

# Mapeo de columnas de EMG para los archivos de trial
COLUMN_MAP_TRIAL = {
    'trial_5.csv': 'R BICEPS BRACHII: EMG 1 [V]', 
    'trial_6.csv': 'L BICEPS BRACHII: EMG 4 [V]',
    'trial_9.csv': 'R DELTOID ANTERIOR: EMG 2 [V]',
    'trial_10.csv': 'L DELTOID ANTERIOR: EMG 3 [V]',
    'trial_11.csv': 'R DELTOID POSTERIOR: EMG 7 [V]',
    'trial_12.csv': 'L DELTOID POSTERIOR: EMG 5 [V]'
}

# Mapeo de nombres de músculo y nombres de archivo MVC esperado
MUSCLE_MAP = {
    'trial_5.csv': ('R BICEPS', 'R_BICEPS', COLUMN_MAP_TRIAL['trial_5.csv']),
    'trial_6.csv': ('L BICEPS', 'L_BICEPS', COLUMN_MAP_TRIAL['trial_6.csv']),
    'trial_9.csv': ('R DELT ANT', 'R_DELTOID_ANTERIOR', COLUMN_MAP_TRIAL['trial_9.csv']),
    'trial_10.csv': ('L DELT ANT', 'L_DELTOID_ANTERIOR', COLUMN_MAP_TRIAL['trial_10.csv']),
    'trial_11.csv': ('R DELT POST', 'R_DELTOID_POSTERIOR', COLUMN_MAP_TRIAL['trial_11.csv']),
    'trial_12.csv': ('L DELT POST', 'L_DELTOID_POSTERIOR', COLUMN_MAP_TRIAL['trial_12.csv'])
}

# --- FUNCIONES DE FILTRADO Y MÉTRICAS ---

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut, highcut, fs):
    if data.size == 0: return np.array([])
    b, a = butter_bandpass(lowcut, highcut, fs)
    try:
        return filtfilt(b, a, data)
    except ValueError:
        # Esto ocurre a menudo si la señal es demasiado corta para el filtro.
        # Devuelve la señal original como un fallback si es necesario, 
        # pero es más seguro devolver array vacío o manejar el error.
        return np.array([])

def rms(signal):
    # Devuelve np.nan si la señal está vacía o es esencialmente cero
    if signal.size == 0 or np.allclose(signal, 0): 
        return np.nan
    return np.sqrt(np.mean(signal**2))

def iemg(signal):
    return np.trapz(np.abs(signal)) if signal.size > 0 else np.nan

def calculate_psd(signal, fs):
    """ nperseg dinámico para evitar advertencias. """
    if signal.size == 0: return np.array([]), np.array([])
    # Aseguramos que nperseg sea par para FFT, si es posible.
    nperseg_val = min(signal.size, int(fs)) 
    nperseg_val = nperseg_val if nperseg_val % 2 == 0 else nperseg_val - 1
    if nperseg_val < 2: return np.array([]), np.array([])

    f, Pxx = welch(signal, fs, window='hann', nperseg=nperseg_val, scaling='density')
    return f, Pxx

def mnf(signal, fs):
    f, Pxx = calculate_psd(signal, fs)
    if Pxx.size == 0 or np.sum(Pxx) == 0: return np.nan
    return np.sum(f * Pxx) / np.sum(Pxx)

def mdf(signal, fs):
    f, Pxx = calculate_psd(signal, fs)
    if Pxx.size == 0 or np.sum(Pxx) == 0: return np.nan
    total_power = np.sum(Pxx)
    cumsum = np.cumsum(Pxx)
    median_index = np.where(cumsum >= total_power * 0.5)[0]
    return f[median_index[0]] if median_index.size > 0 else np.nan

# --- NUEVA FUNCIÓN AUXILIAR: CALCULAR MAX RMS EN VENTANA ---
def max_rms_of_window(signal, fs, window_s=0.5):
    """Calcula el valor RMS máximo de una señal usando una ventana rodante (ej. 500 ms)."""
    window_size = int(fs * window_s)
    
    # Si la señal es más corta que la ventana, solo calcula el RMS de la señal completa.
    if window_size == 0 or len(signal) < window_size:
        return rms(signal) 

    # Implementación eficiente del RMS rodante
    # Usamos la señal rectificada al cuadrado para simplificar la ventana deslizante
    squared_signal = signal**2
    # Convolución con una ventana unitaria para obtener la suma de los cuadrados
    window = np.ones(window_size)
    sum_of_squares = convolve(squared_signal, window, mode='valid')
    
    # Dividir por el tamaño de la ventana y tomar la raíz cuadrada para obtener el RMS
    rms_envelope = np.sqrt(np.maximum(0, sum_of_squares / window_size))
            
    return np.max(rms_envelope) if rms_envelope.size > 0 else np.nan

# --- FUNCIONES DE SEGMENTACIÓN Y EXTRACCIÓN ---

def segment_emg_by_rms_envelope(filtered_signal, fs, window_size_ms, threshold_percent):
    window_size_samples = int(fs * (window_size_ms / 1000.0))
    if window_size_samples == 0 or len(filtered_signal) < window_size_samples:
        return []
    
    # Usar la misma lógica de convolución para la envoltura RMS del trial
    squared_signal = filtered_signal**2
    window = np.ones(window_size_samples)/window_size_samples
    rms_envelope = np.sqrt(np.maximum(0, convolve(squared_signal, window, mode='valid')))
    
    if rms_envelope.size == 0: return []
    
    max_rms_env = np.max(rms_envelope)
    
    if max_rms_env < 1e-6:
        return []
    
    threshold = max_rms_env * (threshold_percent / 100.0)
    
    is_active = rms_envelope > threshold
    diff_active = np.diff(np.pad(is_active, (1, 0), 'constant', constant_values=0).astype(int))

    starts_env = np.where(diff_active == 1)[0]
    stops_env = np.where(diff_active == -1)[0]
    
    # Corrección del offset para mapear al índice original de la señal filtrada
    offset = window_size_samples - 1 
    starts = [s + offset for s in starts_env]
    stops = [s + offset for s in stops_env]

    if stops and starts and stops[0] < starts[0]: starts = starts[1:]
    if starts and stops and starts[-1] > stops[-1]: stops = stops[:-1]

    segments = []
    min_segment_len = 100 
    for start, stop in zip(starts, stops):
        if stop > start and (stop - start) > min_segment_len: 
            segments.append(filtered_signal[start:stop])

    return segments

def extract_features_per_segment(emg_segments, fs):
    segment_features = []
    for segment in emg_segments:
        segment_features.append([rms(segment), iemg(segment), mnf(segment, fs), mdf(segment, fs)])
    
    if len(segment_features) == 0:
        return np.array([np.nan, np.nan, np.nan, np.nan])
        
    return np.nanmean(segment_features, axis=0)

# --- FUNCIÓN PARA EXTRAER RMS_MVC (CORREGIDA) ---

def get_mvc_rms(subject_id, muscle_name_full, column_name_trial):
    """
    Calcula el RMS máximo (MVC) a partir del archivo MVC específico usando la ventana rodante.
    """
    
    # 1. Determinar el nombre base del músculo para el archivo MVC
    if 'BICEPS' in muscle_name_full:
         if 'R_BICEPS' in muscle_name_full:
             mvc_muscle_base = 'R_BICEPS'
         elif 'L_BICEPS' in muscle_name_full:
             mvc_muscle_base = 'L_BICEPS'
         else:
             mvc_muscle_base = 'BICEPS'
             
    elif 'DELTOID' in muscle_name_full:
         mvc_muscle_base = muscle_name_full 
    else:
         mvc_muscle_base = muscle_name_full

    # 2. Construcción de la ruta del archivo MVC
    muscle_file_name = f'subject_{subject_id}_MVC_{mvc_muscle_base}.csv'
    mvc_path = os.path.join(f'sEMG_data/subject_{subject_id}/MVC', muscle_file_name)
    
    # 3. Intento de lectura y procesamiento
    if not os.path.exists(mvc_path):
        if 'BICEPS' in mvc_muscle_base:
            muscle_file_name_alt = f'subject_{subject_id}_MVC_BICEPS.csv'
            mvc_path_alt = os.path.join(f'sEMG_data/subject_{subject_id}/MVC', muscle_file_name_alt)
            if os.path.exists(mvc_path_alt):
                mvc_path = mvc_path_alt
            else:
                return np.nan
        else:
            return np.nan

    try:
        mvc_data = pd.read_csv(mvc_path, delimiter=',', header=0)
        
        # Buscar la columna EMG en el archivo MVC
        muscle_name_prefix = column_name_trial.split(':')[0].strip()
        valid_cols = [col for col in mvc_data.columns if muscle_name_prefix in col and '[V]' in col and 'filter' not in col]
        
        if not valid_cols:
            return np.nan
            
        emg_signal_mvc = mvc_data[valid_cols[0]].dropna().values

        if emg_signal_mvc.size == 0:
            return np.nan
        
        # Filtrar la señal MVC
        filtered_signal_mvc = bandpass_filter(emg_signal_mvc, LOW_CUT, HIGH_CUT, FS)
        
        # CORRECCIÓN: Usar el Max RMS en ventana rodante para MVC
        return max_rms_of_window(filtered_signal_mvc, FS, window_s=MVC_WINDOW_S)
        
    except Exception as e:
        # Aquí puedes dejar un print para depuración si lo necesitas, pero lo quitaré para la salida limpia
        # print(f"Error al procesar el archivo MVC: {mvc_path}. Error: {e}")
        return np.nan

# --- FUNCIÓN PRINCIPAL DE PROCESAMIENTO POR SUJETO ---

def process_subject(subject_id, mvc_cache):
    subject_dir = f'sEMG_data/subject_{subject_id}'
    subject_results = []
    
    # 1. Obtener y cachear todos los MVC para el sujeto
    if subject_id not in mvc_cache:
        mvc_cache[subject_id] = {}
        for file_name, (short_name, full_name, col_name) in MUSCLE_MAP.items():
            # Intentamos obtener el MVC
            mvc_value = get_mvc_rms(subject_id, full_name, col_name)
            mvc_cache[subject_id][file_name] = mvc_value

    for file_name in TRIAL_FILES:
        file_path = os.path.join(subject_dir, file_name)
        short_name, full_name, emg_col_to_analyze = MUSCLE_MAP.get(file_name)
        
        # Inicializar características a NaN
        rms_norm_avg = np.nan
        iemg_avg = np.nan
        mnf_avg = np.nan
        mdf_avg = np.nan
        
        # 1. Obtener el RMS_MVC (normalizador)
        mvc_rms = mvc_cache[subject_id].get(file_name, np.nan)
        
        if not os.path.exists(file_path) or emg_col_to_analyze is None:
            pass
        else:
            try:
                data = pd.read_csv(file_path, delimiter=',', header=0)
                emg_signal = data[emg_col_to_analyze].dropna().values
                
                if emg_signal.size > 0:
                    filtered_signal = bandpass_filter(emg_signal, LOW_CUT, HIGH_CUT, FS)
                    
                    # Segmentación y extracción de características ABSOLUTAS
                    emg_segments = segment_emg_by_rms_envelope(filtered_signal, FS, WINDOW_SIZE_MS, ACTIVATION_THRESHOLD)
                    avg_features_abs = extract_features_per_segment(emg_segments, FS)
                    
                    rms_abs_avg = avg_features_abs[0]
                    iemg_avg = avg_features_abs[1]
                    mnf_avg = avg_features_abs[2]
                    mdf_avg = avg_features_abs[3]
                    
                    # 2. NORMALIZACIÓN RMS (si MVC está disponible)
                    if not np.isnan(rms_abs_avg) and not np.isnan(mvc_rms) and mvc_rms > 0:
                        rms_norm_avg = (rms_abs_avg / mvc_rms) * 100
                    
            except Exception as e:
                # print(f"Error al procesar trial {file_name} de Sujeto_{subject_id}: {e}")
                pass
        
        # Crear la fila de resultados para el DataFrame
        row = [f'Subject_{subject_id}', file_name, rms_norm_avg, iemg_avg, mnf_avg, mdf_avg]
        subject_results.append(row)

    df = pd.DataFrame(subject_results, columns=['Subject', 'Trial', 'RMS (Avg) [% MVC]', 'iEMG (Avg) [V.s]', 'MNF (Avg) [Hz]', 'MDF (Avg) [Hz]'])
    return df

# --- EJECUCIÓN MAESTRA ---

ALL_SUBJECTS_DATA = []
# Asumo que tienes 13 sujetos basado en el código previo.
NUM_SUBJECTS = 13 
subject_1_df = None
mvc_cache = {} 

print("Iniciando procesamiento de datos y normalización MVC...")

for i in range(1, NUM_SUBJECTS + 1):
    subject_df = process_subject(i, mvc_cache)
    ALL_SUBJECTS_DATA.append(subject_df)
    
    if i == 1:
        subject_1_df = subject_df
        
# 1. Concatenar todos los resultados en un solo DataFrame
final_df = pd.concat(ALL_SUBJECTS_DATA, ignore_index=True)

# 2. Guardar el archivo CSV con todos los resultados
final_df.to_csv("EMG_All_Subjects_Features_Normalized.csv", index=False)


# 3. Mostrar la tabla de resultados para Subject_1 (Verificación)
if subject_1_df is not None:
    muscle_map_display = {k: v[0] for k, v in MUSCLE_MAP.items()}
    
    display_df = subject_1_df.copy().set_index('Trial').drop('Subject', axis=1)
    display_df.index = [f'{idx} ({muscle_map_display.get(idx)})' for idx in display_df.index]
    
    print("\n" + "="*100)
    print(" RESULTADOS FINALES: Características Promedio Normalizadas (Sujeto 1 - Verificación)")
    print("(Todos los resultados han sido guardados en 'EMG_All_Subjects_Features_Normalized.csv')")
    print("="*100)
    print(display_df.to_string(float_format="{:.4f}".format)) 
    print("="*100)

print(f"\nArchivo 'EMG_All_Subjects_Features_Normalized.csv' creado con los datos de {NUM_SUBJECTS} sujetos.")

Iniciando procesamiento de datos y normalización MVC...

 RESULTADOS FINALES: Características Promedio Normalizadas (Sujeto 1 - Verificación)
(Todos los resultados han sido guardados en 'EMG_All_Subjects_Features_Normalized.csv')
                            RMS (Avg) [% MVC]  iEMG (Avg) [V.s]  MNF (Avg) [Hz]  MDF (Avg) [Hz]
trial_5.csv (R BICEPS)                28.6864            0.9684         74.3549         67.8597
trial_6.csv (L BICEPS)                29.4323            0.2506         69.6405         60.0547
trial_9.csv (R DELT ANT)               5.1445            0.1263         79.8419         63.1563
trial_10.csv (L DELT ANT)              8.1330            0.1225         80.8750         59.7730
trial_11.csv (R DELT POST)            17.0138            0.0657         63.5096         55.8308
trial_12.csv (L DELT POST)            32.3492            0.0519         60.0495         48.9638

Archivo 'EMG_All_Subjects_Features_Normalized.csv' creado con los datos de 13 sujetos.


In [ ]:
import pandas as pd
import numpy as np
import os
import re 

# --- CONFIGURACIÓN Y CONSTANTES ---
# La ubicación base de los archivos de fatiga.
FATIGUE_DIR_BASE = 'Fatigue_index'
NUM_SUBJECTS = 13 

# Mapeo de trials que SÍ queremos procesar (5, 6, 9, 10, 11, 12)
TRIAL_MAP_FILENAMES = {
    'trial_5.csv': 'R BICEPS', 'trial_6.csv': 'L BICEPS',
    'trial_9.csv': 'R DELT ANT', 'trial_10.csv': 'L DELT ANT',
    'trial_11.csv': 'R DELT POST', 'trial_12.csv': 'L DELT POST'
}

# Nombres de los archivos
EMG_FEATURES_FILE = "EMG_All_Subjects_Features_Normalized.csv"
FATIGUE_OUTPUT_FILE = "Fatigue_Perception_Times_Subset.csv"
MERGED_OUTPUT_FILE = "EMG_Fatigue_Analysis_Subset.csv"

# --- FUNCIONES PARA EL ANÁLISIS DE FATIGA AUTOPERCIBIDA ---

def find_transition_time(data, start_label, end_label):
    """
    Encuentra el tiempo (en segundos) de la primera transición de start_label a end_label.
    """
    if data.empty:
        return np.nan

    # La transición ocurre cuando la etiqueta anterior era 'start_label' 
    # y la etiqueta actual es 'end_label'.
    # Usamos .copy() para evitar SettingWithCopyWarning en .shift()
    temp_data = data.copy()
    transition = (temp_data['label'].shift(1) == start_label) & (temp_data['label'] == end_label)
    
    first_transition_index = transition[transition].index
    
    if not first_transition_index.empty:
        # Devuelve el valor de 'time' correspondiente al *primer* instante de transición
        return data.loc[first_transition_index[0], 'time']
    else:
        return np.nan

def process_fatigue_data(subject_id, trial_map_filenames):
    """
    Procesa los archivos de fatiga para un sujeto y extrae los tiempos de transición.
    """
    subject_fatigue_times = []
    
    for trial_file_key in trial_map_filenames.keys():
        
        # Extraemos el número de trial (ej: de 'trial_5.csv' extraemos '5')
        match = re.search(r'trial_(\d+)\.csv', trial_file_key)
        if not match:
            continue
            
        trial_num = match.group(1) 
        
        # Construcción de la ruta del archivo de fatiga (ej: 'Trial_5.csv')
        fatigue_file_name = f'Trial_{trial_num}.csv'
        fatigue_path = os.path.join(FATIGUE_DIR_BASE, f'subject_{subject_id}', fatigue_file_name)
        
        time_to_fatigue_1 = np.nan
        time_to_fatigue_2 = np.nan
        
        if os.path.exists(fatigue_path):
            try:
                # Leer el archivo
                fatigue_data = pd.read_csv(fatigue_path, header=0, delimiter=',')
                
                # Asegurar que las columnas existan y renombrar para estandarizar
                time_col = [col for col in fatigue_data.columns if 'time' in col.lower()]
                label_col = [col for col in fatigue_data.columns if 'label' in col.lower()]
                
                if time_col and label_col:
                    fatigue_data = fatigue_data[[time_col[0], label_col[0]]]
                    fatigue_data.columns = ['time', 'label']
                    
                    # Extraer tiempos de transición
                    time_to_fatigue_1 = find_transition_time(fatigue_data, 0, 1)
                    time_to_fatigue_2 = find_transition_time(fatigue_data, 1, 2)
                
            except Exception:
                # Silenciamos errores menores si no se puede procesar un archivo
                pass
        
        # Añadir los resultados al listado
        subject_fatigue_times.append([
            f'Subject_{subject_id}', 
            trial_file_key, 
            time_to_fatigue_1, 
            time_to_fatigue_2
        ])

    return pd.DataFrame(subject_fatigue_times, 
                         columns=['Subject', 'Trial', 'Time_Fatigue_Level_1 [s]', 'Time_Fatigue_Level_2 [s]'])

# --- EJECUCIÓN MAESTRA ---

# 1. Extracción de Tiempos de Fatiga
print("="*80)
print("INICIANDO EXTRACCIÓN DE TIEMPOS DE FATIGA")
print("="*80)

FATIGUE_DATA_ALL = []
for i in range(1, NUM_SUBJECTS + 1):
    fatigue_df = process_fatigue_data(i, TRIAL_MAP_FILENAMES)
    FATIGUE_DATA_ALL.append(fatigue_df)

final_fatigue_df = pd.concat(FATIGUE_DATA_ALL, ignore_index=True)

# Guardar el archivo CSV de tiempos de fatiga
final_fatigue_df.to_csv(FATIGUE_OUTPUT_FILE, index=False)
print(f"\n Tiempos de fatiga guardados en '{FATIGUE_OUTPUT_FILE}'.")


# 2. Unificación de Datos EMG y Fatiga
print("\n" + "="*80)
print("INICIANDO UNIFICACIÓN DE DATOS EMG Y TIEMPOS DE FATIGA")
print("="*80)
try:
    # Cargar el archivo de características EMG normalizadas
    emg_df = pd.read_csv(EMG_FEATURES_FILE)
    
    # Fusionar los DataFrames por 'Subject' y 'Trial'. 
    merged_df = pd.merge(emg_df, final_fatigue_df, on=['Subject', 'Trial'], how='left')
    
    # Guardar el archivo unificado
    merged_df.to_csv(MERGED_OUTPUT_FILE, index=False)
    
    print(f" Datos de EMG y Fatiga Unificados y guardados en '{MERGED_OUTPUT_FILE}'.")
    
    # Mostrar el resultado unificado (formato de salida requerido)
    print("\n--- SALIDA REQUERIDA: MUESTRA DEL ARCHIVO UNIFICADO ---")
    display_cols = ['Subject', 'Trial', 'RMS (Avg) [% MVC]', 'iEMG (Avg) [V.s]', 'MNF (Avg) [Hz]', 'MDF (Avg) [Hz]', 'Time_Fatigue_Level_1 [s]', 'Time_Fatigue_Level_2 [s]']
    
    # Aseguramos el orden y formato de salida
    print(merged_df[display_cols].head(12).to_string(float_format="{:.4f}".format, index=False))
    print("...")

except FileNotFoundError:
    print(f" ERROR: No se encontró el archivo de características EMG: '{EMG_FEATURES_FILE}'. Por favor, asegúrate de que el archivo exista.")
except Exception as e:
    print(f" ERROR al intentar fusionar los datos: {e}")

INICIANDO EXTRACCIÓN DE TIEMPOS DE FATIGA

✅ Tiempos de fatiga guardados en 'Fatigue_Perception_Times_Subset.csv'.

INICIANDO UNIFICACIÓN DE DATOS EMG Y TIEMPOS DE FATIGA
✅ Datos de EMG y Fatiga Unificados y guardados en 'EMG_Fatigue_Analysis_Subset.csv'.

--- SALIDA REQUERIDA: MUESTRA DEL ARCHIVO UNIFICADO ---
  Subject        Trial  RMS (Avg) [% MVC]  iEMG (Avg) [V.s]  MNF (Avg) [Hz]  MDF (Avg) [Hz]  Time_Fatigue_Level_1 [s]  Time_Fatigue_Level_2 [s]
Subject_1  trial_5.csv            28.6864            0.9684         74.3549         67.8597                   62.4200                  120.9800
Subject_1  trial_6.csv            29.4323            0.2506         69.6405         60.0547                  111.4800                  165.6790
Subject_1  trial_9.csv             5.1445            0.1263         79.8419         63.1563                  123.8790                  211.9390
Subject_1 trial_10.csv             8.1330            0.1225         80.8750         59.7730                  28